In [ ]:
import boto3, sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker import image_uris
from sagemaker.processing import ProcessingInput, ProcessingOutput
import os
ACCOUNT_ID = os.getenv("AWS_ACCOUNT_ID") #replace with your 12-digit aws account ID
REGION = os.getenv("AWS_REGION", "us-east-1") 
ROLE_ARN   = f"arn:aws:iam::{ACCOUNT_ID}:role/ThreatDetectionSageMakerRole"
PROJECT    = "threat-detection"
RAW_S3     = f"s3://{PROJECT}-logs-processed-{ACCOUNT_ID}/processed/"
ARTS_S3    = f"s3://{PROJECT}-model-artifacts-{ACCOUNT_ID}/"

sess = sagemaker.Session()

input_data  = ParameterString(name="InputData",      default_value=RAW_S3)
instance_tp = ParameterString(name="TrainInstance",  default_value="ml.m5.xlarge")
min_acc     = ParameterFloat( name="MinAccuracy",    default_value=0.85)

sklearn_processor = SKLearnProcessor(
    framework_version = "1.2-1",
    instance_type     = "ml.t3.medium",
    instance_count    = 1,
    role              = ROLE_ARN,
    sagemaker_session = sess
)

preprocess_step = ProcessingStep(
    name      = "PreprocessThreatLogs",
    processor = sklearn_processor,
    inputs    = [ProcessingInput(source=input_data, destination="/opt/ml/processing/input")],
    outputs   = [
        ProcessingOutput(output_name="train",
                         source="/opt/ml/processing/output/train",
                         destination=f"{ARTS_S3}data/train/"),
        ProcessingOutput(output_name="validation",
                         source="/opt/ml/processing/output/validation",
                         destination=f"{ARTS_S3}data/validation/")
    ],
    code = "preprocess.py"
)

xgboost_image = image_uris.retrieve(
    framework="xgboost", region=REGION, version="1.7-1", image_scope="training"
)

xgb_estimator = Estimator(
    image_uri         = xgboost_image,
    instance_type     = instance_tp,
    instance_count    = 1,
    output_path       = f"{ARTS_S3}model/",
    role              = ROLE_ARN,
    sagemaker_session = sess,
    hyperparameters   = {
        "objective":        "binary:logistic",
        "num_round":        "200",
        "max_depth":        "6",
        "eta":              "0.2",
        "subsample":        "0.8",
        "colsample_bytree": "0.8",
        "eval_metric":      "auc",
        "scale_pos_weight": "10"
    }
)

train_step = TrainingStep(
    name      = "TrainXGBoost",
    estimator = xgb_estimator,
    inputs    = {
        "train": TrainingInput(
            s3_data=preprocess_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=preprocess_step.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

register_step = RegisterModel(
    name                     = "RegisterThreatModel",
    estimator                = xgb_estimator,
    model_data               = train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types            = ["text/csv"],
    response_types           = ["application/json"],
    inference_instances      = ["ml.m5.large", "ml.m5.xlarge"],
    transform_instances      = ["ml.m5.xlarge"],
    model_package_group_name = "ThreatDetectionModels",
    approval_status          = "Approved"
)

pipeline = Pipeline(
    name              = "ThreatDetectionPipeline",
    parameters        = [input_data, instance_tp, min_acc],
    steps             = [preprocess_step, train_step, register_step],
    sagemaker_session = sess
)

pipeline.upsert(role_arn=ROLE_ARN)
print("Pipeline upserted. Starting execution...")
execution = pipeline.start()
print(f"Execution ARN: {execution.arn}")
execution.wait()
print("Pipeline complete.")